In [0]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import numpy as np

In [0]:
ds = pd.read_csv('/Workspace/Users/w10168315@usm.edu/HonorsThesis/final_dataset.csv')

In [0]:
print(ds)

In [0]:
print(ds.columns)

### Removed rows with null/empty "text" values

In [0]:
# Set this to the name of your text column (e.g. 'text', 'comment', 'content')
text_col = 'text'

if text_col not in ds.columns:
    print(f"'{text_col}' not found. Available columns: {list(ds.columns)}")

else:
    before = len(ds)
    ds = ds[ds[text_col].notna() & (ds[text_col].astype(str).str.strip() != '')].reset_index(drop=True)
    print(f"Removed {before - len(ds)} rows with empty '{text_col}'. {len(ds)} rows remain.")

### Check the count of each column

In [0]:
print(ds['label'].isnull().sum())
print(ds['attack_type'].isnull().sum())
print(ds['is_synthetic'].isnull().sum())

In [0]:
text_col = 'text'

# --- 1. Exact duplicates (raw text) ---

exact_dup_mask = ds[text_col].duplicated(keep=False) # For each row, returns True if its text appears >1 anywhere in ds; keep=false means it also counts the original.

print(f"Exact duplicate rows (including originals): {exact_dup_mask.sum()}") # Counts all rows whose text is duplicated, counting every copy

print(f"Distinct duplicated texts: {ds.loc[exact_dup_mask, text_col].nunique()}") # Among flagged rows, counts how many unique text values are actually involved

print(ds[exact_dup_mask].groupby(text_col).size().sort_values(ascending=False).head(10)) # Groups duplicated rows by their text & times each appears, and show the top 10 ones



# --- 2. Duplicates after normalization (whitespace, case, punctuation) ---
def normalize(t):
    t = str(t).lower() #lowercase everything
    t = re.sub(r'[^a-z0-9\s]', ' ', t) #replaces characters (not lowercase letter, digit, or whitespace) with a space
    t = re.sub(r'\s+', ' ', t).strip() #collapses multiple spaces into one and trims the ends
    return t

ds['_text_norm'] = ds[text_col].apply(normalize) #applies normalize function to every text, storing results in a temporary helper column
norm_dup_mask = ds['_text_norm'].duplicated(keep=False) #duplicate detection on the normalized texts
print(f"\nNormalized duplicate rows: {norm_dup_mask.sum()}") #total count
print(ds[norm_dup_mask].groupby('_text_norm').size().sort_values(ascending=False).head(10)) #groupBy



# --- 3. Near duplicates (TF-IDF + cosine similarity) ---
SAMPLE_SIZE = 2000
if len(ds) > SAMPLE_SIZE: #if ds big, anlyz random sampl 2,000-42(reproducible)
    sample = ds.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"\nDataset is large; comparing a random sample of {SAMPLE_SIZE} rows for near duplicates.")

else:
    sample = ds.reset_index(drop=True)

vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5)) #converts each normalized text into a numeric vector
tfidf = vectorizer.fit_transform(sample['_text_norm'])

sim_matrix = cosine_similarity(tfidf) # computes an n×n matrix where entry (i, j) is the cosine similarity between texts i and j: 1.0 = identical direction, 0.0 = nothing in common.


np.fill_diagonal(sim_matrix, 0)

SIM_THRESHOLD = 0.85 #pairs scoring >= 0.85 are near duplicates

# list comprehension loops over all row pairs & keep those meeting threshold, along with scores.
near_pairs = [
    (i, j, sim_matrix[i, j])
    for i in range(len(sample))
    for j in range(i + 1, len(sample))
    if sim_matrix[i, j] >= SIM_THRESHOLD
]

print(f"\nNear-duplicate pairs (cosine similarity >= {SIM_THRESHOLD}): {len(near_pairs)}")

for i, j, score in near_pairs[:20]:
    print(f"\nSim: {score:.3f}")
    print(f"  Row {i}: {sample.loc[i, text_col][:100]}")
    print(f"  Row {j}: {sample.loc[j, text_col][:100]}")

# Drop helper column
ds = ds.drop(columns=['_text_norm'])